In [10]:
import sqlite3
import pandas as pd

# Connect to your "filing cabinet"
conn = sqlite3.connect('nfl_data.db')

# This query finds the top 10 QBs by average EPA per dropback in 2024
query = """
SELECT passer, AVG(epa) as avg_epa
FROM pbp_data
WHERE season = 2024 AND play_type = 'pass' AND passer IS NOT NULL
GROUP BY passer
ORDER BY avg_epa DESC
LIMIT 10
"""

df = pd.read_sql_query(query, conn)
print(df)


           passer   avg_epa
0          A.Cole  4.333918
1           T.Way  3.984215
2        C.Sutton  3.745369
3           J.Fox  3.637429
4     J.Jefferson  3.050907
5         S.Diggs  2.762919
6      A.Mitchell  2.445659
7       N.Mullens  2.434363
8  J.Smith-Njigba  2.243609
9       St. Brown  1.724116


In [8]:
# This tells you the names of all the columns in your table
df_columns = pd.read_sql_query("SELECT name FROM pragma_table_info('pbp_data')", conn)
print(df_columns)


                  name
0              play_id
1              game_id
2          old_game_id
3            home_team
4            away_team
..                 ...
391      defense_names
392  offense_positions
393  defense_positions
394    offense_numbers
395    defense_numbers

[396 rows x 1 columns]


In [19]:
# Use pbp_data instead of a non-existent 'games' table
query = """
SELECT 
    stadium, 
    SUM(home_wins) as total_home_wins,
    SUM(away_wins) as total_away_wins
FROM (
    SELECT 
        game_id, 
        stadium,
        MAX(home_score) as home_score,
        MAX(away_score) as away_score,
        CASE WHEN MAX(home_score) > MAX(away_score) THEN 1 ELSE 0 END as home_wins,
        CASE WHEN MAX(away_score) > MAX(home_score) THEN 1 ELSE 0 END as away_wins
    FROM pbp_data
    WHERE season = 2025
    GROUP BY game_id, stadium
)
GROUP BY stadium
ORDER BY total_home_wins DESC
"""

df_wins = pd.read_sql_query(query, conn)
print(df_wins)

                            stadium  total_home_wins  total_away_wins
0                      SoFi Stadium               12                4
1        Empower Field at Mile High                9                2
2                       Lumen Field                8                2
3                  Gillette Stadium                8                3
4                     Soldier Field                7                3
5                       NRG Stadium                7                2
6                  Highmark Stadium                7                2
7                  EverBank Stadium                7                2
8                   MetLife Stadium                5               11
9                 Lucas Oil Stadium                5                3
10          Lincoln Financial Field                5                4
11                  Levi's® Stadium                5                4
12                    Lambeau Field                5                3
13  GEHA Field at Ar

In [11]:

query = """
SELECT
    play_type,
    COUNT(*) as total_plays,
    AVG(epa) as avg_epa,
    AVG(yards_gained) as avg_yards
FROM pbp_data
WHERE posteam = 'IND'
    AND season = 2025
    AND down IN (1,2)
    AND play_type IN ('run','pass')
GROUP BY play_type
"""

df_colts = pd.read_sql_query(query, conn)
print("Colts Early Down Efficiency (2025):")
print(df_colts)


Colts Early Down Efficiency (2025):
  play_type  total_plays   avg_epa  avg_yards
0      pass          423  0.043757   6.520095
1       run          361  0.051760   4.775623


In [15]:
query = """
SELECT 
    CASE WHEN week <= 8 THEN 'Early (1-8)' ELSE 'Late (9-13)' END as season_segment,
    play_type, 
    COUNT(*) as total_plays, 
    AVG(epa) as avg_epa, 
    AVG(yards_gained) as avg_yards 
FROM pbp_data 
WHERE posteam = 'IND' 
  AND season = 2025 
  AND down IN (1,2) 
  AND play_type IN ('run', 'pass') 
GROUP BY season_segment, play_type
ORDER BY season_segment, play_type
"""

df_colts_split = pd.read_sql_query(query, conn)
print("Colts Early Down Efficiency: Early Weeks vs. Rest of Season")
print(df_colts_split)

Colts Early Down Efficiency: Early Weeks vs. Rest of Season
  season_segment play_type  total_plays   avg_epa  avg_yards
0    Early (1-8)      pass          196  0.248581   7.474490
1    Early (1-8)       run          169  0.153409   5.544379
2    Late (9-13)      pass          227 -0.133096   5.696035
3    Late (9-13)       run          192 -0.037713   4.098958


In [ ]:
SELECT 
    query = """
    SELECT
        CASE
            WHEN week <= 8 THEN 'Weeks 1-8'
            WHEN week BETWEEN 9 AND 13 THEN 'Weeks 9-13'
            ELSE 'Other'
        END AS season_segment,
        SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) AS run_plays,
        SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) AS pass_plays,
        CAST(SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) AS FLOAT) /
        NULLIF(SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END), 0) AS run_to_pass_ratio
    FROM pbp_data
    WHERE posteam = 'IND'
      AND season = 2025
      AND down IN (1, 2)
      AND play_type IN ('run', 'pass')
      AND week <= 13
    GROUP BY season_segment
    ORDER BY season_segment;
    """

    df_season_segments = pd.read_sql_query(query, conn)
    print(df_season_segments)
        WHEN week <= 8 THEN 'Weeks 1-8' 
        WHEN week BETWEEN 9 AND 13 THEN 'Weeks 9-13' 
        ELSE 'Other' 
    END as season_segment,
    SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) as run_plays,
    SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) as pass_plays,
    CAST(SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) AS FLOAT) / 
    NULLIF(SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END), 0) as run_to_pass_ratio
FROM pbp_data
WHERE posteam = 'IND' 
  AND season = 2025
  AND down IN (1, 2)
  AND play_type IN ('run', 'pass')
  AND week <= 13
GROUP BY season_segment
ORDER BY season_segment;

IndentationError: unexpected indent (1621052719.py, line 2)